In [20]:
# ==============================
# PREREQ: Build ZCTA centroids + valid ZIP whitelist
# ==============================
import os
import geopandas as gpd
import pandas as pd

# Try common locations you showed in your tree
CANDIDATE_DIRS = [
    "../data/tl_2020_us_zcta510"     # repo root
]
ZCTA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(d)), None)
assert ZCTA_DIR is not None, f"ZCTA shapefile dir not found in: {CANDIDATE_DIRS}"

zcta = gpd.read_file(ZCTA_DIR).to_crs(epsg=4326)
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else (
         "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else None)
assert id_col is not None, f"Could not find ZCTA id column in {list(zcta.columns)[:10]}"

zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x
zcta["lat"] = zcta["centroid"].y

zip_centroids = zcta[[id_col, "lat", "lon"]].rename(columns={id_col:"zip"}).copy()
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)

# Whitelist of valid U.S. ZIPs (ZCTAs)
valid_us_zips = set(zip_centroids["zip"].tolist())
print("zip_centroids:", zip_centroids.shape, "| valid_us_zips:", len(valid_us_zips))


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80390/2393523475.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


zip_centroids: (33144, 3) | valid_us_zips: 33144


In [21]:
# ==============================
# COMBINE CSVs (your columns) + dedup by NCT + build US-only site ZIPs
# ==============================
import os, re, warnings, glob
import numpy as np
import pandas as pd

DATA_DIR = "../data"
FILES = [
    "Colorectal.csv",
    "Pancreatic Cancer.csv",
    "Gastric Cancer.csv",
    "Esophageal cancer.csv",
    "Hepatocellular.csv",
    "cholangiocarcinoma.csv",
]

def _norm(s): return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

# Map filename -> cancer key
F2KEY = {
    _norm("Colorectal"):               "colorectal_cancer",
    _norm("Pancreatic Cancer"):        "pancreatic_cancer",
    _norm("Gastric Cancer"):           "gastric_cancer",
    _norm("Esophageal cancer"):        "oesophageal_cancer",
    _norm("Hepatocellular"):           "hepatocellular_cancer",
    _norm("cholangiocarcinoma"):       "cholangiocarcinoma",
}

# --- Phase normalization (Early Phase 1 → Phase 1; 1/2 → Phase 2; 2/3 → Phase 3)
def normalize_phase(x: str) -> str | None:
    if not isinstance(x, str): return None
    t = x.strip().lower()
    if "early phase 1" in t or "phase 0" in t: return "Phase 1"
    if re.search(r"\bphase\s*1\b", t) and re.search(r"\bphase\s*2\b", t): return "Phase 2"
    if re.search(r"\bphase\s*2\b", t) and re.search(r"\bphase\s*3\b", t): return "Phase 3"
    if re.search(r"\bphase\s*1\b", t): return "Phase 1"
    if re.search(r"\bphase\s*2\b", t): return "Phase 2"
    if re.search(r"\bphase\s*3\b", t): return "Phase 3"
    if re.search(r"\bphase\s*4\b", t): return "Phase 4"
    return None

# --- US detection + ZIP extraction (guard against non-US 5-digit numbers)
US_COUNTRY_PAT = re.compile(r"\b(united\s*states|u\.s\.a\.?|u\.s\.)\b", re.I)
US_STATE_ABBRS = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
    "ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"
}
US_STATE_NAMES = {
    "alabama","alaska","arizona","arkansas","california","colorado","connecticut","delaware",
    "district of columbia","florida","georgia","hawaii","idaho","illinois","indiana","iowa",
    "kansas","kentucky","louisiana","maine","maryland","massachusetts","michigan","minnesota",
    "mississippi","missouri","montana","nebraska","nevada","new hampshire","new jersey",
    "new mexico","new york","north carolina","north dakota","ohio","oklahoma","oregon",
    "pennsylvania","rhode island","south carolina","south dakota","tennessee","texas","utah",
    "vermont","virginia","washington","west virginia","wisconsin","wyoming"
}
ZIP5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")

def is_us_site(s: str) -> bool:
    if not isinstance(s, str): return False
    low = s.lower()
    if US_COUNTRY_PAT.search(low): return True
    # ", ST" pattern (state abbrev)
    m = re.findall(r",\s*([A-Z]{2})\b", s)
    if any(x in US_STATE_ABBRS for x in m): return True
    # spelled-out state
    if any(name in low for name in US_STATE_NAMES): return True
    return False

def zip_list_from_token(s: str):
    if not isinstance(s, str): return []
    return ZIP5_RE.findall(s)

frames = []
sites_rows = []
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        warnings.warn(f"Missing file: {p}")
        continue

    df = pd.read_csv(p, dtype=str)

    # Rename your exact columns -> normalized names used below
    rename_map = {
        "NCT Number":   "nct_id",
        "Study Title":  "title",
        "Conditions":   "conditions",
        "Interventions":"interventions",
        "Phases":       "phase_raw",
        "Funder Type":  "funder_type",
        "Locations":    "locations",
    }
    # also tolerate lower/upper variants
    for k,v in list(rename_map.items()):
        if k not in df.columns and k.lower() in [c.lower() for c in df.columns]:
            real = [c for c in df.columns if c.lower()==k.lower()][0]
            rename_map[real] = v
            rename_map.pop(k, None)

    df = df.rename(columns=rename_map)
    required = {"nct_id","locations","phase_raw"}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{f}: missing required columns {missing}")

    # tag group from filename
    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")
    df["cancer_type"] = key

    # normalize phase
    df["phase_norm"] = df["phase_raw"].apply(normalize_phase)

    # keep minimal master cols (dedup by NCT within group)
    frames.append(df[["nct_id","cancer_type","phase_norm"]].copy())

    # US site ZIPs (explode)
    tok = df[["nct_id","locations"]].copy()
    tok["loc_list"] = tok["locations"].astype(str).str.split(r"\s*\|\s*")
    tok = tok.explode("loc_list", ignore_index=True).dropna(subset=["loc_list"])
    tok = tok[tok["loc_list"].apply(is_us_site)]
    tok["zip"] = tok["loc_list"].apply(lambda s: zip_list_from_token(s)[:1]).str[0]
    tok["zip"] = tok["zip"].astype(str).str.zfill(5)
    # keep only real US ZCTAs
    tok = tok[tok["zip"].isin(valid_us_zips)]
    tok["cancer_type"] = key
    tok = tok.drop_duplicates(subset=["nct_id","zip","cancer_type"])
    sites_rows.append(tok[["nct_id","cancer_type","zip"]])

# Master trials (dedup by NCT within group)
master = (pd.concat(frames, ignore_index=True)
          if frames else pd.DataFrame(columns=["nct_id","cancer_type","phase_norm"]))
master = master.sort_values(["nct_id","cancer_type"]).drop_duplicates(subset=["nct_id","cancer_type"], keep="first")

# US site ZIPs (validated by ZCTA whitelist)
site_df = (pd.concat(sites_rows, ignore_index=True)
           if sites_rows else pd.DataFrame(columns=["nct_id","cancer_type","zip"]))
site_df["zip"] = site_df["zip"].astype(str).str.zfill(5)
# also join lat/lon if you want immediately:
trial_sites = site_df.drop_duplicates(["zip","cancer_type"]).merge(zip_centroids, on="zip", how="left").dropna(subset=["lat","lon"])

print("Master trials:", master.shape)
print("Site rows (US-only ZIPs):", site_df.shape)
print("Unique geocoded sites:", len(trial_sites))
master.head(3), site_df.head(3)


Master trials: (958, 3)
Site rows (US-only ZIPs): (11164, 3)
Unique geocoded sites: 4471


(          nct_id        cancer_type phase_norm
 220  NCT01174121  colorectal_cancer    Phase 2
 487  NCT01174121  pancreatic_cancer    Phase 2
 294  NCT01954992  pancreatic_cancer    Phase 3,
         nct_id        cancer_type    zip
 0  NCT05806931  colorectal_cancer  07202
 1  NCT05806931  colorectal_cancer  07302
 2  NCT05806931  colorectal_cancer  08701)

In [22]:
# ==============================
# Phase 1/2/3 counts & enrollment (by cancer group)
# ==============================
import numpy as np
import pandas as pd

assert {"nct_id","cancer_type","phase_norm"}.issubset(master.columns), \
    "master must have columns: nct_id, cancer_type, phase_norm"

# If you later add an enrollment column to master, name it "enrollment_num" (float).
has_enroll = "enrollment_num" in master.columns

phase_order = ["Phase 1","Phase 2","Phase 3"]

def _agg(group: pd.DataFrame) -> pd.Series:
    out = {}
    for ph in phase_order:
        m = group["phase_norm"] == ph
        out[f"Trials {ph}"] = int(group.loc[m, "nct_id"].nunique())
        if has_enroll:
            out[f"Enrollment {ph}"] = float(group.loc[m, "enrollment_num"].dropna().sum())
    return pd.Series(out)

phase_summary = master.groupby("cancer_type", dropna=False).apply(_agg).reset_index()
phase_summary.rename(columns={"cancer_type":"Cancer Key"}, inplace=True)

# Optional: pretty labels
LABELS = {
    "colorectal_cancer":"Colorectal",
    "pancreatic_cancer":"Pancreatic",
    "gastric_cancer":"Gastric",
    "oesophageal_cancer":"Esophageal",
    "hepatocellular_cancer":"Liver (HCC)",
    "cholangiocarcinoma":"Cholangiocarcinoma",
}
phase_summary.insert(0, "Cancer Type", phase_summary["Cancer Key"].map(LABELS).fillna(phase_summary["Cancer Key"]))

# Save + show
out_dir = "out"; os.makedirs(out_dir, exist_ok=True)
phase_summary.to_csv(os.path.join(out_dir, "phase_summary.csv"), index=False)
phase_summary


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80390/3956418600.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  phase_summary = master.groupby("cancer_type", dropna=False).apply(_agg).reset_index()


,Cancer Type,Cancer Key,Trials Phase 1,Trials Phase 2,Trials Phase 3
0,Cholangiocarcinoma,cholangiocarcinoma,24,52,5
1,Colorectal,colorectal_cancer,117,149,15
2,Gastric,gastric_cancer,58,68,9
3,Liver (HCC),hepatocellular_cancer,33,67,7
4,Esophageal,oesophageal_cancer,36,46,4
5,Pancreatic,pancreatic_cancer,98,132,19


In [23]:
# ==============================
# Build ZIP/ZCTA universe (RUCA + ACS) -> zip_universe
# ==============================
import os, re, requests, numpy as np, pandas as pd

assert "zip_centroids" in globals(), "Run the PREREQ cell that builds `zip_centroids` first."

# ---- Load RUCA (classify Urban/Rural)
ruca_path = os.path.join("../data", "RUCA-codes-2020-zipcode.csv")
ruca_raw = pd.read_csv(ruca_path, dtype=str)

def _norm(s): return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")
ruca = ruca_raw.rename(columns={c: _norm(c) for c in ruca_raw.columns})

# Guess primary RUCA code column
cand_ruca_cols = [c for c in ruca.columns if ("primary" in c and "ruca" in c)] \
              or [c for c in ruca.columns if c.endswith("ruca") or c.startswith("ruca")]
assert cand_ruca_cols, f"Could not find a RUCA code column in {list(ruca.columns)[:10]}"
ruca_code_col = cand_ruca_cols[0]

# Guess ZIP column (the one that contains mostly 5-digit strings)
zip_col = None
for c in ruca.columns:
    m = ruca[c].astype(str).str.extract(r"(\d{5})")[0]
    if m.notna().mean() >= 0.9:
        zip_col = c; break
assert zip_col, "Could not find a ZIP column in RUCA CSV."

def _ruca_to_urb(x: str) -> str:
    try:
        v = float(str(x).strip())
    except Exception:
        return "Unknown"
    if 1.0 <= v <= 3.0:  return "Urban"
    if 4.0 <= v <= 10.0: return "Rural"
    return "Unknown"

ruca_df = pd.DataFrame({
    "zip": ruca[zip_col].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5),
    "PrimaryRUCA": ruca[ruca_code_col].astype(str)
}).dropna(subset=["zip"]).drop_duplicates(subset=["zip"])
ruca_df["rural_urban"] = ruca_df["PrimaryRUCA"].apply(_ruca_to_urb)

# ---- ACS fetch helpers (2023)
YEAR = 2023
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "").strip()

def fetch_acs_vars(year: int, vars_list: list[str]) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": ",".join(["NAME"] + vars_list), "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()
    cols = rows[0]
    data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area":"zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    for v in vars_list:
        df[v] = pd.to_numeric(df[v], errors="coerce")
    return df[["zip"] + vars_list]

# Population, income, ethnicity (Hispanic), race (Black)
pop_df = fetch_acs_vars(YEAR, ["B01003_001E"]).rename(columns={"B01003_001E":"population"})
inc    = fetch_acs_vars(YEAR, ["B19013_001E"]).rename(columns={"B19013_001E":"mhi"})
eth    = fetch_acs_vars(YEAR, ["B03003_001E","B03003_003E"]).rename(
            columns={"B03003_001E":"pop_total_eth","B03003_003E":"pop_hispanic"})
blk    = fetch_acs_vars(YEAR, ["B02001_001E","B02001_003E"]).rename(
            columns={"B02001_001E":"pop_total_race","B02001_003E":"pop_black"})

# ---- Assemble universe
zip_universe = (
    zip_centroids
      .merge(ruca_df[["zip","rural_urban"]], on="zip", how="left")
      .merge(pop_df, on="zip", how="left")
      .merge(inc,    on="zip", how="left")
      .merge(eth,    on="zip", how="left")
      .merge(blk,    on="zip", how="left")
)

# Ensure numerics
for c in ["population","mhi","pop_total_eth","pop_hispanic","pop_total_race","pop_black"]:
    if c in zip_universe.columns:
        zip_universe[c] = pd.to_numeric(zip_universe[c], errors="coerce")

print("ZIP universe rows:", len(zip_universe),
      "| pop present:", f"{100*zip_universe['population'].notna().mean():.1f}%",
      "| mhi present:", f"{100*zip_universe['mhi'].notna().mean():.1f}%")

# ---- Income quartiles (population-weighted)
def weighted_quantile(values, weights, qs):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w) - 0.5*w
    cw = cw / np.sum(w)
    return np.interp(qs, cw, v)

mask = zip_universe["mhi"].notna() & zip_universe["population"].notna()
if mask.any():
    q1,q2,q3 = weighted_quantile(zip_universe.loc[mask,"mhi"],
                                 zip_universe.loc[mask,"population"], [0.25,0.5,0.75])
    zip_universe["inc_quartile"] = pd.cut(zip_universe["mhi"],
                                          bins=[-np.inf,q1,q2,q3,np.inf],
                                          labels=["Q1 (Lowest)","Q2","Q3","Q4 (Highest)"])
else:
    zip_universe["inc_quartile"] = np.nan

zip_universe.head(3)


ZIP universe rows: 33144 | pop present: 99.3% | mhi present: 99.3%


,zip,lat,lon,rural_urban,population,mhi,pop_total_eth,pop_hispanic,pop_total_race,pop_black,inc_quartile
0,43451,41.312827,-83.615185,Urban,1003.0,63333.0,1003.0,72.0,1003.0,93.0,Q2
1,43452,41.515177,-82.966801,Urban,14052.0,63815.0,14052.0,702.0,14052.0,322.0,Q2
2,43456,41.668369,-82.822768,Rural,749.0,86250.0,749.0,16.0,749.0,53.0,Q3


In [24]:
# ==============================
# Repair: add `phase_raw` back to `master` and re-normalize phases
# ==============================
import os, re, pandas as pd, numpy as np

# --- Expect these from earlier cells; define if missing
try:
    DATA_DIR
except NameError:
    DATA_DIR = "data"

try:
    FILES
except NameError:
    FILES = [
        "Colorectal.csv",
        "Pancreatic Cancer.csv",
        "Gastric Cancer.csv",
        "Esophageal cancer.csv",
        "Hepatocellular.csv",
        "cholangiocarcinoma.csv",
    ]

def _norm(s): return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")
try:
    F2KEY
except NameError:
    F2KEY = {
        _norm("Colorectal"):               "colorectal_cancer",
        _norm("Pancreatic Cancer"):        "pancreatic_cancer",
        _norm("Gastric Cancer"):           "gastric_cancer",
        _norm("Esophageal cancer"):        "oesophageal_cancer",
        _norm("Hepatocellular"):           "hepatocellular_cancer",
        _norm("cholangiocarcinoma"):       "cholangiocarcinoma",
    }

# --- Build a (nct_id, cancer_type, phase_raw) table from the raw CSVs
rows = []
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p, dtype=str)

    # find columns case-insensitively
    nct_col   = next((c for c in df.columns if c.lower().startswith("nct")), None)
    phases_col= next((c for c in df.columns if c.lower()=="phases"), None)
    if nct_col is None or phases_col is None:
        continue

    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")
    tmp = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "cancer_type": key,
        "phase_raw": df[phases_col].astype(str)
    })
    rows.append(tmp)

phase_raw_df = (pd.concat(rows, ignore_index=True)
                if rows else pd.DataFrame(columns=["nct_id","cancer_type","phase_raw"]))
phase_raw_df = phase_raw_df.dropna(subset=["nct_id"]).drop_duplicates(["nct_id","cancer_type"], keep="first")

# --- Attach to master
if "master" not in globals():
    raise RuntimeError("`master` not found — run the combine step first.")
master = master.drop(columns=["phase_raw"], errors="ignore").merge(
    phase_raw_df, on=["nct_id","cancer_type"], how="left"
)

# --- Phase normalization with roman numerals & combos
ROMAN = {"i":1,"ii":2,"iii":3,"iv":4}

def _norm_phase_token(t: str) -> int | None:
    if not isinstance(t, str): return None
    s = t.strip().lower()

    if "early phase 1" in s or "phase 0" in s:  # PI rule: merge into Phase 1
        return 1

    # capture "phase i", "phase 1b", "phase ii a", "phase 2/3", etc.
    hits = re.findall(r"phase\s*([ivx]+|\d+)\s*[ab]?", s)
    if not hits:
        # fallback: bare numerals/romans like "ii/iii"
        hits = re.findall(r"\b([ivx]+|\d+)\b", s)

    vals = []
    for g in hits:
        g = g.lower()
        if g.isdigit():
            vals.append(int(g))
        else:
            vals.append(ROMAN.get(g))
    vals = [v for v in vals if v in {1,2,3,4}]
    if not vals:
        return None

    # PI rule for combos
    if set(vals) == {1,2}: return 2
    if set(vals) == {2,3}: return 3
    return max(vals)

def normalize_phase(s: str) -> str | None:
    v = _norm_phase_token(s)
    return f"Phase {v}" if v else None

# Fill only where missing / "None"
mask = master["phase_norm"].isna() | (master["phase_norm"].astype(str).str.lower() == "none")
if "phase_raw" in master.columns:
    master.loc[mask, "phase_norm"] = master.loc[mask, "phase_raw"].apply(normalize_phase)
else:
    raise RuntimeError("Could not find a raw phase column to normalize (phase_raw).")

# --- Show result
print(master["phase_norm"].value_counts(dropna=False).to_string())


phase_norm
Phase 2    514
Phase 1    385
Phase 3     59


In [25]:
# ==============================
# Coverage @ 30 & 60 mi + stratified tables + exports
# Requires: master, trial_sites, zip_universe already built
# ==============================
import os, numpy as np, pandas as pd
from sklearn.neighbors import BallTree

# --- sanity guards
for need in ["master", "trial_sites", "zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`; run the earlier cells that build it.")

# Ensure numeric
zip_universe["population"] = pd.to_numeric(zip_universe["population"], errors="coerce")
if "mhi" in zip_universe:
    zip_universe["mhi"] = pd.to_numeric(zip_universe["mhi"], errors="coerce")

EARTH_RADIUS_MI = 3958.7613
RADII_MI = [30, 60]

# --- label map
LABELS = {
    "any_gi":"Any GI",
    "colorectal_cancer":"Colorectal",
    "pancreatic_cancer":"Pancreatic",
    "gastric_cancer":"Gastric",
    "oesophageal_cancer":"Esophageal",
    "hepatocellular_cancer":"Liver (HCC)",
    "cholangiocarcinoma":"Cholangiocarcinoma",
}

# --- build BallTrees
def build_tree(df: pd.DataFrame):
    pts = np.deg2rad(df[["lat","lon"]].to_numpy())
    return BallTree(pts, metric="haversine")

trees = {}
if len(trial_sites):
    for ctype, g in trial_sites.groupby("cancer_type"):
        trees[ctype] = build_tree(g)
    tree_any = build_tree(trial_sites)
else:
    tree_any = None

qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

# --- coverage flags for each radius
for r in RADII_MI:
    r_rad = r / EARTH_RADIUS_MI
    # pooled Any GI
    zip_universe[f"covered_any_gi_{r}mi"] = False if tree_any is None else \
        np.array([len(ix)>0 for ix in tree_any.query_radius(qpts, r=r_rad)], dtype=bool)
    # per cancer group
    for ctype in sorted(master["cancer_type"].unique()):
        t = trees.get(ctype)
        col = f"covered_{ctype}_{r}mi"
        zip_universe[col] = False if t is None else \
            np.array([len(ix)>0 for ix in t.query_radius(qpts, r=r_rad)], dtype=bool)

# --- income quartiles if missing
def weighted_quantile(values, weights, qs):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w) - 0.5*w
    cw = cw / np.sum(w)
    return np.interp(qs, cw, v)

if "inc_quartile" not in zip_universe.columns:
    mask = zip_universe["mhi"].notna() & zip_universe["population"].notna()
    if mask.any():
        q1,q2,q3 = weighted_quantile(zip_universe.loc[mask,"mhi"],
                                     zip_universe.loc[mask,"population"], [0.25,0.5,0.75])
        zip_universe["inc_quartile"] = pd.cut(zip_universe["mhi"],
                                              bins=[-np.inf,q1,q2,q3,np.inf],
                                              labels=["Q1 (Lowest)","Q2","Q3","Q4 (Highest)"])
    else:
        zip_universe["inc_quartile"] = np.nan

# --- helpers
def pop_share(df, covered_col, weight_col="population"):
    if covered_col not in df or weight_col not in df or len(df)==0: return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def subpop_share(df, covered_col, subpop_col):
    if covered_col not in df or subpop_col not in df: return float("nan")
    w = pd.to_numeric(df[subpop_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def table_for_radius(r):
    rows = []
    keys = ["any_gi"] + list(master["cancer_type"].unique())
    for key in keys:
        label = LABELS.get(key, key)
        col = f"covered_any_gi_{r}mi" if key=="any_gi" else f"covered_{key}_{r}mi"
        pop_all  = pop_share(zip_universe, col)
        pop_urb  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Urban"], col)
        pop_rur  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Rural"], col)
        q1       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q1 (Lowest)"], col)
        q4       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q4 (Highest)"], col)
        hisp     = subpop_share(zip_universe, col, "pop_hispanic")
        black    = subpop_share(zip_universe, col, "pop_black")
        rows.append({
            "Cancer Type": label,
            "% Pop Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
            "% Urban Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
            "% Rural Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
            "% Q1 Income Covered": 100*q1 if pd.notna(q1) else np.nan,
            "% Q4 Income Covered": 100*q4 if pd.notna(q4) else np.nan,
            "% Hispanic Covered": 100*hisp if pd.notna(hisp) else np.nan,
            "% Black Covered": 100*black if pd.notna(black) else np.nan,
        })
    return pd.DataFrame(rows)

table_30 = table_for_radius(30)
table_60 = table_for_radius(60)

# --- export all
out_dir = "out"; os.makedirs(out_dir, exist_ok=True)
table_30.to_csv(os.path.join(out_dir, "coverage_30mi.csv"), index=False)
table_60.to_csv(os.path.join(out_dir, "coverage_60mi.csv"), index=False)

# if you built `phase_summary` earlier, save it alongside in one Excel
xlsx_path = os.path.join(out_dir, "GI_coverage_and_phase_summary.xlsx")
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as w:
    if 'phase_summary' in globals():
        phase_summary.to_excel(w, sheet_name="Phase Summary", index=False)
    table_30.to_excel(w, sheet_name="Coverage 30mi", index=False)
    table_60.to_excel(w, sheet_name="Coverage 60mi", index=False)

print("Saved:")
print(" -", os.path.join(out_dir, "coverage_30mi.csv"))
print(" -", os.path.join(out_dir, "coverage_60mi.csv"))
if 'phase_summary' in globals():
    print(" -", xlsx_path)
else:
    print(" - (Excel workbook saved without Phase Summary) ->", xlsx_path)


Saved:
 - out/coverage_30mi.csv
 - out/coverage_60mi.csv
 - out/GI_coverage_and_phase_summary.xlsx


In [26]:
# Any GI, 60 mi — sanity peek
table_60.loc[table_60["Cancer Type"]=="Any GI"]


,Cancer Type,% Pop Covered,% Urban Covered,% Rural Covered,% Q1 Income Covered,% Q4 Income Covered,% Hispanic Covered,% Black Covered
0,Any GI,94.630469,96.721401,84.222215,87.08765,99.374759,91.349302,95.215829


In [27]:
# ==============================
# GI Trials QC Battery — US ZIPs, Locations, Duplicates, Phases, Coverage sanity
# ==============================
import os, re, pandas as pd, numpy as np
os.makedirs("out", exist_ok=True)

# ---- Guards
need = ["zip_centroids", "valid_us_zips", "master", "site_df", "trial_sites", "zip_universe", "DATA_DIR", "FILES"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}. Run the earlier cells first.")

# ---- Helpers (same logic you used for extraction)
US_COUNTRY_PAT = re.compile(r"\b(united\s*states|u\.s\.a\.?|u\.s\.)\b", re.I)
US_STATE_ABBRS = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
    "ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"
}
US_STATE_NAMES = {
    "alabama","alaska","arizona","arkansas","california","colorado","connecticut","delaware",
    "district of columbia","florida","georgia","hawaii","idaho","illinois","indiana","iowa",
    "kansas","kentucky","louisiana","maine","maryland","massachusetts","michigan","minnesota",
    "mississippi","missouri","montana","nebraska","nevada","new hampshire","new jersey",
    "new mexico","new york","north carolina","north dakota","ohio","oklahoma","oregon",
    "pennsylvania","rhode island","south carolina","south dakota","tennessee","texas","utah",
    "vermont","virginia","washington","west virginia","wisconsin","wyoming"
}
ZIP5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")

COMMON_NONUS_COUNTRIES = [
    "germany","france","italy","brazil","chile","china","taiwan","norway","switzerland","korea",
    "japan","spain","portugal","sweden","canada","mexico","argentina","australia","netherlands","poland",
]

def is_us_site(s: str) -> bool:
    if not isinstance(s, str): return False
    low = s.lower()
    if US_COUNTRY_PAT.search(low): return True
    m = re.findall(r",\s*([A-Z]{2})\b", s)
    if any(x in US_STATE_ABBRS for x in m): return True
    if any(name in low for name in US_STATE_NAMES): return True
    return False

def extract_tokens_from_files(files):
    """Explode Locations text into individual site tokens (no ZIP filter)."""
    rows = []
    for f in files:
        p = os.path.join(DATA_DIR, f)
        if not os.path.exists(p): 
            continue
        df = pd.read_csv(p, dtype=str)
        # tolerate casing
        loc_col = "Locations" if "Locations" in df.columns else ("Location" if "Location" in df.columns else None)
        nct_col = "NCT Number" if "NCT Number" in df.columns else next((c for c in df.columns if c.lower().startswith("nct")), None)
        if not loc_col or not nct_col:
            continue
        tok = df[[nct_col, loc_col]].rename(columns={nct_col:"nct_id", loc_col:"loc"}).copy()
        tok["nct_id"] = tok["nct_id"].astype(str).str.strip()
        tok["loc_list"] = tok["loc"].astype(str).str.split(r"\s*\|\s*")
        tok = tok.explode("loc_list", ignore_index=True).dropna(subset=["loc_list"])
        tok["file"] = f
        rows.append(tok[["file","nct_id","loc_list"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["file","nct_id","loc_list"])

# ---------- QC 0: Site_df basics
print("\n[QC0] site_df basic integrity")
site_df["zip"] = site_df["zip"].astype(str).str.zfill(5)
bad_len = (site_df["zip"].str.len() != 5).sum()
not_in_whitelist = (~site_df["zip"].isin(valid_us_zips)).sum()
dups = site_df.duplicated(subset=["nct_id","zip","cancer_type"]).sum()
print(f" - rows: {len(site_df):,} | unique (NCT,ZIP,group): {len(site_df.drop_duplicates(['nct_id','zip','cancer_type'])):,}")
print(f" - zip length ≠5: {bad_len} | zip not in ZCTA whitelist: {not_in_whitelist} | duplicates: {dups}")

if bad_len or not_in_whitelist or dups:
    site_df.loc[site_df["zip"].str.len()!=5].to_csv("out/qc_bad_zip_length.csv", index=False)
    site_df.loc[~site_df["zip"].isin(valid_us_zips)].to_csv("out/qc_zip_not_in_whitelist.csv", index=False)
    site_df.loc[site_df.duplicated(subset=["nct_id","zip","cancer_type"], keep=False)].to_csv("out/qc_duplicates_nct_zip_group.csv", index=False)
    print("   -> wrote details under out/qc_*.csv")

# ---------- QC 1: Re-scan raw Locations & flag non-US tokens
print("\n[QC1] Raw Locations scan (non-US tokens check)")
tokens = extract_tokens_from_files(FILES)
if tokens.empty:
    print(" - No tokens parsed (check FILES list / columns).")
else:
    tokens["is_us"] = tokens["loc_list"].apply(is_us_site)
    tokens["has_zip5"] = tokens["loc_list"].str.contains(ZIP5_RE)
    tokens["looks_foreign"] = tokens["loc_list"].str.lower().apply(lambda s: any(c in s for c in COMMON_NONUS_COUNTRIES))
    share_us = tokens["is_us"].mean() if len(tokens) else float("nan")
    share_foreign = tokens["looks_foreign"].mean() if len(tokens) else float("nan")
    print(f" - tokens total: {len(tokens):,} | US-like: {share_us:.1%} | mentions non-US country: {share_foreign:.1%}")
    # Non-US examples (top 20) and any US-flagged tokens that still mention foreign countries
    non_us = tokens[~tokens["is_us"]]
    foreign_in_us = tokens[tokens["is_us"] & tokens["looks_foreign"]]
    if len(non_us):
        non_us.head(50).to_csv("out/qc_non_us_token_examples.csv", index=False)
        print(f"   -> wrote out/qc_non_us_token_examples.csv (sample {min(50,len(non_us))} rows)")
    if len(foreign_in_us):
        foreign_in_us.head(50).to_csv("out/qc_foreign_mentions_but_us_flag.csv", index=False)
        print(f"   -> wrote out/qc_foreign_mentions_but_us_flag.csv (sample {min(50,len(foreign_in_us))} rows)")

# ---------- QC 2: Link tokens → zips like your extractor, then verify all zips are real US ZCTAs
print("\n[QC2] Token-to-ZIP audit (re-derive ZIPs from tokens and validate against ZCTA)")
if not tokens.empty:
    audit = tokens.copy()
    audit["zip"] = audit["loc_list"].apply(lambda s: ZIP5_RE.findall(s)[:1]).str[0]
    audit = audit.dropna(subset=["zip"])
    audit["zip"] = audit["zip"].astype(str).str.zfill(5)
    audit["zip_in_whitelist"] = audit["zip"].isin(valid_us_zips)
    bad = audit[~audit["zip_in_whitelist"]]
    print(f" - token-derived ZIPs: {len(audit):,} | not-in-whitelist: {len(bad):,}")
    if len(bad):
        bad.to_csv("out/qc_token_zips_not_in_whitelist.csv", index=False)
        print("   -> wrote out/qc_token_zips_not_in_whitelist.csv")
else:
    print(" - skipped (no tokens)")

# ---------- QC 3: Trial site geocoding completeness
print("\n[QC3] Geocoding completeness (trial_sites has lat/lon via ZCTA)")
missing_geo = trial_sites["lat"].isna().sum() + trial_sites["lon"].isna().sum()
print(f" - trial_sites rows: {len(trial_sites):,} | missing lat/lon fields (sum): {missing_geo}")
if missing_geo:
    trial_sites.loc[trial_sites["lat"].isna() | trial_sites["lon"].isna()].to_csv("out/qc_sites_missing_geo.csv", index=False)
    print("   -> wrote out/qc_sites_missing_geo.csv")

# ---------- QC 4: RUCA / ACS completeness in zip_universe
print("\n[QC4] zip_universe data completeness")
for col in ["population","mhi","pop_hispanic","pop_total_eth","pop_black","pop_total_race","rural_urban","inc_quartile"]:
    if col in zip_universe.columns:
        miss = zip_universe[col].isna().mean()
        print(f" - {col}: missing {miss:.1%}")
# list zips without RUCA classification
if "rural_urban" in zip_universe.columns and zip_universe["rural_urban"].isna().any():
    zip_universe[zip_universe["rural_urban"].isna()].head(100).to_csv("out/qc_missing_ruca_sample.csv", index=False)
    print("   -> wrote out/qc_missing_ruca_sample.csv (sample)")

# ---------- QC 5: Phase normalization sanity
print("\n[QC5] Phase normalization sanity")
phase_counts = master["phase_norm"].value_counts(dropna=False)
print(" - phase_norm counts:\n", phase_counts.to_string())
if master["phase_norm"].isna().any():
    master[master["phase_norm"].isna()].head(50).to_csv("out/qc_phase_unknown_sample.csv", index=False)
    print("   -> wrote out/qc_phase_unknown_sample.csv (sample trials with unmapped phases)")

# ---------- QC 6: Highly recurrent ZIPs (could be HQ/PO boxes)
print("\n[QC6] Top ZIPs by site frequency (flag potential HQs)")
top_zip = site_df["zip"].value_counts().head(20)
print(top_zip.to_string())
top_zip.to_csv("out/qc_top_site_zips.csv", header=["site_rows"], index_label="zip")

# ---------- QC 7: Coverage monotonicity (if you already built 30/60 mi flags)
print("\n[QC7] Coverage monotonicity 30→60 mi (if available)")
cov30 = [c for c in zip_universe.columns if c.endswith("_30mi")]
cov60 = [c for c in zip_universe.columns if c.endswith("_60mi")]
if cov30 and cov60:
    probs = []
    for c30 in cov30:
        c60 = c30.replace("_30mi","_60mi")
        if c60 in zip_universe:
            bad_mask = zip_universe[c30] & ~zip_universe[c60]
            if bad_mask.any():
                probs.append((c30, int(bad_mask.sum())))
    if probs:
        print(" - PROBLEM: some zips covered at 30mi but not at 60mi:")
        for c30, n in probs:
            print(f"   {c30} → {c30.replace('_30mi','_60mi')}: {n} zips")
    else:
        print(" - OK: all 60mi coverages are supersets of 30mi")
else:
    print(" - 30/60 flags not found; skip")

print("\nQC complete. Any detailed findings were written to the out/ folder as qc_*.csv")



[QC0] site_df basic integrity
 - rows: 11,164 | unique (NCT,ZIP,group): 11,164
 - zip length ≠5: 0 | zip not in ZCTA whitelist: 0 | duplicates: 0

[QC1] Raw Locations scan (non-US tokens check)
 - tokens total: 22,037 | US-like: 59.7% | mentions non-US country: 33.0%
   -> wrote out/qc_non_us_token_examples.csv (sample 50 rows)
   -> wrote out/qc_foreign_mentions_but_us_flag.csv (sample 50 rows)

[QC2] Token-to-ZIP audit (re-derive ZIPs from tokens and validate against ZCTA)
 - token-derived ZIPs: 16,688 | not-in-whitelist: 2,831
   -> wrote out/qc_token_zips_not_in_whitelist.csv

[QC3] Geocoding completeness (trial_sites has lat/lon via ZCTA)
 - trial_sites rows: 4,471 | missing lat/lon fields (sum): 0

[QC4] zip_universe data completeness
 - population: missing 0.7%
 - mhi: missing 0.7%
 - pop_hispanic: missing 0.7%
 - pop_total_eth: missing 0.7%
 - pop_black: missing 0.7%
 - pop_total_race: missing 0.7%
 - rural_urban: missing 0.1%
 - inc_quartile: missing 0.7%
   -> wrote out/qc_m

/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80390/3506829470.py:88: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tokens["has_zip5"] = tokens["loc_list"].str.contains(ZIP5_RE)


In [28]:
import hashlib, json, pandas as pd, os
def sha(p): 
    h=hashlib.sha256(); 
    with open(p,'rb') as f: h.update(f.read()); 
    return h.hexdigest()
os.makedirs("out", exist_ok=True)
master.to_csv("out/master.csv", index=False)
site_df.to_csv("out/site_df.csv", index=False)
zip_universe.to_parquet("out/zip_universe.parquet", index=False)
manifest = {
  "acs_year": 2023, "ruca_file": "data/RUCA-codes-2020-zipcode.csv",
  "radii_miles": [30,60],
  "inputs": {k: sha(v) for k,v in {
      "master":"out/master.csv","site_df":"out/site_df.csv","zip_universe":"out/zip_universe.parquet"
  }.items()}
}
with open("out/manifest.json","w") as f: json.dump(manifest,f,indent=2)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [29]:
# Random manual audit
import numpy as np
np.random.seed(7)
audit_us  = site_df.sample(min(50, len(site_df)), random_state=7)
audit_us.to_csv("out/audit_us_site_rows.csv", index=False)
print("Saved 50 US site rows to inspect: out/audit_us_site_rows.csv")


Saved 50 US site rows to inspect: out/audit_us_site_rows.csv


In [30]:
# Delta vs. strict US clinic filter (from earlier message)
def coverage_table_from(trial_sites_local, label, r=60):
    from sklearn.neighbors import BallTree
    EARTH_RADIUS_MI=3958.7613
    trees = {k: BallTree(np.deg2rad(v[["lat","lon"]].to_numpy()), metric="haversine")
             for k,v in trial_sites_local.groupby("cancer_type")}
    tree_any = BallTree(np.deg2rad(trial_sites_local[["lat","lon"]].to_numpy()), metric="haversine")
    qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())
    def covered(col_prefix, tree):
        r_rad = r/EARTH_RADIUS_MI
        mask = np.array([len(ix)>0 for ix in tree.query_radius(qpts, r=r_rad)], bool)
        return (zip_universe["population"].fillna(0)*mask).sum()/zip_universe["population"].sum()
    rows=[]
    rows.append({"Cancer Key":"any_gi", "Coverage": covered("any", tree_any)})
    for k,t in trees.items():
        rows.append({"Cancer Key":k, "Coverage": covered(k, t)})
    return pd.DataFrame(rows).assign(Config=label)

# Make a strict version (if you created it); else reuse current
strict_sites = trial_sites.copy()  # replace with your stricter variant if you built it
base = coverage_table_from(trial_sites, "base", r=60)
strict = coverage_table_from(strict_sites, "strict", r=60)
delta = base.merge(strict, on="Cancer Key", suffixes=("_base","_strict"))
delta["Abs Δ (pp)"] = 100*(delta["Coverage_strict"]-delta["Coverage_base"])
delta.sort_values("Abs Δ (pp)", ascending=True)


,Cancer Key,Coverage_base,Config_base,Coverage_strict,Config_strict,Abs Δ (pp)
0,any_gi,0.946305,base,0.946305,strict,0.0
1,cholangiocarcinoma,0.796883,base,0.796883,strict,0.0
2,colorectal_cancer,0.932368,base,0.932368,strict,0.0
3,gastric_cancer,0.904307,base,0.904307,strict,0.0
4,hepatocellular_cancer,0.762161,base,0.762161,strict,0.0
5,oesophageal_cancer,0.847602,base,0.847602,strict,0.0
6,pancreatic_cancer,0.922847,base,0.922847,strict,0.0


In [32]:
# Radius sensitivity
def radius_sensitivity(r_list=[25,30,35,55,60,65]):
    from sklearn.neighbors import BallTree
    EARTH_RADIUS_MI=3958.7613
    tree_any = BallTree(np.deg2rad(trial_sites[["lat","lon"]].to_numpy()), metric="haversine")
    qpts=np.deg2rad(zip_universe[["lat","lon"]].to_numpy())
    rows=[]
    for r in r_list:
        mask=np.array([len(ix)>0 for ix in tree_any.query_radius(qpts, r=r/EARTH_RADIUS_MI)],bool)
        cov=(zip_universe["population"].fillna(0)*mask).sum()/zip_universe["population"].sum()
        rows.append({"Radius(mi)":r,"Any GI %":100*cov})
    return pd.DataFrame(rows)
radius_sensitivity()


,Radius(mi),Any GI %
0,25,83.207122
1,30,86.051563
2,35,88.295832
3,55,93.781305
4,60,94.630469
5,65,95.304161


In [33]:
# Population-weighted bootstrap CI for Any GI 60mi
import numpy as np
mask = zip_universe["covered_any_gi_60mi"].astype(bool).to_numpy()
w    = zip_universe["population"].fillna(0).to_numpy()
def cov_est(mask, w): return (mask*w).sum()/w.sum()
def boot(n=200, rng=42):
    rs = np.random.RandomState(rng); est=[]
    for _ in range(n):
        # Poisson(1) bootstrap weights
        m = rs.poisson(1, size=len(w))
        est.append(cov_est(mask, w*m))
    return np.percentile(np.array(est), [2.5,50,97.5])
ci = boot()
print("Any GI 60mi bootstrap CI (%, 200 reps):", (ci*100).round(2))


Any GI 60mi bootstrap CI (%, 200 reps): [94.27 94.67 94.95]


In [37]:
# ==============================
# Phase 1/2/3: Trials & Enrollment (per group + Any GI single row)
# Reuses `enroll_df` built earlier (nct_id, cancer_type, phase_norm, enrollment_num)
# ==============================
import numpy as np
import pandas as pd

assert {"nct_id","cancer_type","phase_norm","enrollment_num"}.issubset(enroll_df.columns), \
    "Run the previous cell that builds `enroll_df` first."

phase_order = ["Phase 1","Phase 2","Phase 3"]

# ---- 1) Per-group wide table (no FutureWarning; NaN when no numeric enrollment)
# Trial counts by phase
counts = (enroll_df.dropna(subset=["phase_norm"])
          .groupby(["cancer_type","phase_norm"])["nct_id"]
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=phase_order, fill_value=0))

counts.columns = [f"Trials {ph}" for ph in counts.columns]

# Enrollment sums by phase (min_count=1 → NaN if all NaN)
enr = (enroll_df.groupby(["cancer_type","phase_norm"])["enrollment_num"]
       .sum(min_count=1)
       .unstack()
       .reindex(columns=phase_order))

enr.columns = [f"Enrollment {ph}" for ph in enr.columns]

per_group = pd.concat([counts, enr], axis=1).reset_index()

# Pretty labels
LABELS = {
    "colorectal_cancer":"Colorectal",
    "pancreatic_cancer":"Pancreatic",
    "gastric_cancer":"Gastric",
    "oesophageal_cancer":"Esophageal",
    "hepatocellular_cancer":"Liver (HCC)",
    "cholangiocarcinoma":"Cholangiocarcinoma",
}
per_group.insert(0, "Cancer Type", per_group["cancer_type"].map(LABELS).fillna(per_group["cancer_type"]))
per_group = per_group.drop(columns=["cancer_type"])

# Order columns nicely
ordered_cols = ["Cancer Type"]
for ph in phase_order:
    ordered_cols += [f"Trials {ph}", f"Enrollment {ph}"]
per_group = per_group[ordered_cols]

# ---- 2) Any GI (Unique NCT) SINGLE ROW
# Choose one phase per trial: the highest normalized phase observed across groups
def phase_to_int(ph):
    return {"Phase 1":1, "Phase 2":2, "Phase 3":3, "Phase 4":4}.get(ph, np.nan)

gi_unique = (enroll_df
             .assign(phase_int=lambda d: d["phase_norm"].map(phase_to_int))
             .groupby("nct_id", as_index=False)
             .agg(phase_int=("phase_int","max"),
                  enrollment_num=("enrollment_num","max")))

gi_unique["phase_norm"] = gi_unique["phase_int"].map({1:"Phase 1",2:"Phase 2",3:"Phase 3",4:"Phase 4"})

# Counts & enrollment by phase
any_counts = gi_unique.groupby("phase_norm")["nct_id"].nunique()
any_enr    = gi_unique.groupby("phase_norm")["enrollment_num"].sum(min_count=1)

row = {"Cancer Type": "Any GI (Unique NCT)"}
for ph in phase_order:
    row[f"Trials {ph}"]     = int(any_counts.get(ph, 0))
    val = any_enr.get(ph, np.nan)
    row[f"Enrollment {ph}"] = float(val) if pd.notna(val) else np.nan

any_gi_one_row = pd.DataFrame([row])

# ---- 3) Save + display
out_dir = "out"
os.makedirs(out_dir, exist_ok=True)
per_group.to_csv(os.path.join(out_dir, "phase_enrollment_by_group.csv"), index=False)
any_gi_one_row.to_csv(os.path.join(out_dir, "phase_enrollment_any_gi_unique_row.csv"), index=False)

with pd.ExcelWriter(os.path.join(out_dir, "GI_phase_trials_and_enrollment.xlsx"), engine="xlsxwriter") as w:
    per_group.to_excel(w, sheet_name="By Group", index=False)
    any_gi_one_row.to_excel(w, sheet_name="Any GI (Unique NCT)", index=False)

per_group, any_gi_one_row


(          Cancer Type  Trials Phase 1  Enrollment Phase 1  Trials Phase 2  \
 0  Cholangiocarcinoma              24                 NaN              52   
 1          Colorectal             123             20084.0             149   
 2             Gastric              59              9478.0              68   
 3         Liver (HCC)              34                 NaN              67   
 4          Esophageal              36                 NaN              46   
 5          Pancreatic             109             17997.0             132   
 
    Enrollment Phase 2  Trials Phase 3  Enrollment Phase 3  
 0                 NaN               5                 NaN  
 1             23521.0              15             17833.0  
 2             12365.0               9              4686.0  
 3                 NaN               7                 NaN  
 4                 NaN               4                 NaN  
 5             18861.0              19             14362.0  ,
            Cancer Type 